# 사용자 정의 손실 함수 구현하기(Huber Loss)

### 문제 설명
PyTorch에서 **Huber Loss**를 사용자 정의 손실 함수로 구현합니다. Huber loss는 회귀 작업에서 사용되는 견고한 손실 함수로, 평균제곱오차(MSE)보다 이상치에 덜 민감합니다. 이 손실은 임계값 파라미터 $ \delta $ 를 기준으로 L2 손실(제곱 오차)과 L1 손실(절대 오차) 사이를 전환합니다.

Huber loss는 수학적으로 다음과 같이 정의됩니다:
$$
L_{\delta}(y, \hat{y}) = 
\begin{cases} 
\frac{1}{2}(y - \hat{y})^2 & \text{if } |y - \hat{y}| \leq \delta, \\
\delta \cdot (|y - \hat{y}| - \frac{1}{2} \delta) & \text{if } |y - \hat{y}| > \delta,
\end{cases}
$$

여기서:
- $y$ 는 실제 값입니다.
- $\hat{y}$ 는 예측 값입니다.
- $\delta$ 는 L1 손실과 L2 손실 사이의 전환을 제어하는 임계값 파라미터입니다.

### 요구사항
1. **사용자 정의 손실 함수**:
   - `torch.nn.Module`을 상속하는 `HuberLoss` 클래스를 구현합니다.
   - 공식에 따라 Huber loss를 계산하도록 `forward` 메서드를 정의합니다.

2. **회귀 모델에서의 사용**:
   - 사용자 정의 손실 함수를 회귀 학습 파이프라인에 통합합니다.
   - 모델 학습 중 손실을 계산하고 최적화하는 데 사용합니다.

### 제약 사항
- 구현은 $ y $ (실제 값)와 $ \hat{y} $ (예측 값)에 대해 스칼라 입력과 배치 입력을 모두 처리해야 합니다.


추가 설명: https://en.wikipedia.org/wiki/Huber_loss

<details>
  <summary>💡 힌트</summary>
  추가 설명: https://www.kaggle.com/code/bigironsphere/loss-function-library-keras-pytorch/notebook
</details>


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Generate synthetic data
torch.manual_seed(42)
X = torch.rand(100, 1) * 10  # 100 data points between 0 and 10
y = 2 * X + 3 + torch.randn(100, 1)  # Linear relationship with noise

class HuberLoss(nn.Module):
    # nn.Module을 상속받은걸 보니 아무래도 learnable parameter가 존재하거나 꽤 복잡한 torch 함수들을 실행한다고 판단할 수 있다.
    # nn.Module을 활용해 learnable parameter가 존재하는 loss function일수도?
    def __init__(self, delta=1.0):
        super(HuberLoss, self).__init__()
        self.delta = delta
    def forward(self, y_pred, y_true):
        diff = torch.abs(y_pred - y_true)
        huber_loss = torch.where(diff <= self.delta, 0.5 * diff**2, self.delta * (diff - 0.5 * self.delta))
        return torch.mean(huber_loss)

# Define the Linear Regression Model
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super(LinearRegressionModel, self).__init__()
        self.linear = nn.Linear(1, 1)  # Single input and single output

    def forward(self, x):
        return self.linear(x)

# Initialize the model, loss function, and optimizer
model = LinearRegressionModel()
#TODO: Add the loss 
criterion = HuberLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    predictions = model(X)
    loss = criterion(predictions, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")


In [ ]:
# Display the learned parameters
[w, b] = model.linear.parameters()
print(f"Learned weight: {w.item():.4f}, Learned bias: {b.item():.4f}")

# Testing on new data
X_test = torch.tensor([[4.0], [7.0]])
with torch.no_grad():
    predictions = model(X_test)
    print(f"Predictions for {X_test.tolist()}: {predictions.tolist()}")